### Basic HuggingFace API

1. Load Model / Tokenizer.
2. Run Simple Inference
3. Run Simple Training

In [2]:
# Load Model and Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "Qwen/Qwen2-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [44]:
# Run Simple Inference

prompt = "Give me a step-by-step preparation plan for Research Engineer interviews at frontier AI labs."
messages = [
    {"role": "system", "content": "You are a helpful interview preparation coach."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print(model_inputs)
print(model_inputs["input_ids"].shape) # B x L

generated_ids = model.generate(**model_inputs, max_new_tokens=512) # B x Genaration
generated_ids = [
    output_ids[len(input_ids):] for output_ids, input_ids in zip(generated_ids, model_inputs.input_ids)
    ]
print(generated_ids)

response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response_text)


<|im_start|>system
You are a helpful interview preparation coach.<|im_end|>
<|im_start|>user
Give me a step-by-step preparation plan for Research Engineer interviews at frontier AI labs.<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,   7128,  17975,
           7247,     13, 151645,    198, 151644,    872,    198,  35127,    752,
            264,   3019,  14319,  29208,  17975,   3119,    369,   8319,  28383,
          19344,    518,  48000,  15235,  49948,     13, 151645,    198, 151644,
          77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 38])
[tensor([97191,   369,   264,  8319, 28383,  3476,   304, 57318, 15235, 40640,
        11136, 17601,  3807,  1376,  7354,   311,  5978,   498,  2299,  1632,
        21334,  7212,   369,   279, 10916,   323, 35595, 13566,   315,  1493,
      

In [45]:
print(response_text)
next_prompt = "Repeat your first sentence again"
messages.extend(
    [
    {"role": "assistant", "content": response_text},
    {"role": "user", "content": next_prompt}
    ]
)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt = True
)

model_inputs = tokenizer([text], return_tensors="pt")

generated_ids = model.generate(**model_inputs, max_new_tokens=512)
generated_ids = [
    output_ids[len(input_ids):] for output_ids, input_ids in zip(generated_ids, model_inputs.input_ids)
]
response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response_text)


Preparing for a Research Engineer role in Frontier AI Labs typically involves several key steps to ensure you're well-prepared for the technical and behavioral aspects of these interviews:

1. **Understand the Company's Focus**: Before diving into your research, it’s important to understand what Frontier AI Labs does specifically. This includes its mission, values, products, and recent projects. This will help tailor your responses to questions that align with their focus.

2. **Research the Industry**: Familiarize yourself with the current state of AI research and technology trends. Look at publications from leading AI journals, conferences, and think tanks. Understanding the latest developments can demonstrate your understanding of the field.

3. **Review Past Research**: Review past research papers, case studies, or whitepapers related to your area of interest. These could be found on academic databases like Google Scholar or by searching for specific keywords online. Highlighting r

In [52]:
batch_messages = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is backpropagation?"}
    ],
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is gradient clipping?"}
    ]
]

texts = [
    tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True
    )
    for chat in batch_messages
]

model_inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    padding_side='left'
).to(device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=128
)

responses = []
for i in range(len(texts)):
    prompt_len = model_inputs["attention_mask"][i].sum().item()
    new_tokens = generated_ids[i, prompt_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    responses.append(text)

for i, r in enumerate(responses):
    print(f"Response {i}: {r}")

Response 0: Backpropagation is an algorithm used in artificial neural networks to train the network by adjusting the weights of the connections between the neurons. It is a type of gradient descent optimization method that allows the network to learn from its mistakes and improve its performance over time.

The basic idea behind backpropagation is to calculate the error made by the network for each input-output pair, and then propagate this error backward through the network using a chain rule to find the change in weight for each neuron in the network. This process is repeated until the network has learned to make accurate predictions on new data inputs.

There are two main types of backpropagation: vanilla
Response 1: 
Gradient clipping is a technique used in machine learning and deep learning to prevent gradients from exploding or vanishing during training. It involves limiting the magnitude of the gradients at each step, which helps to prevent the model from getting stuck in local 

In [40]:
print(text)

<|im_start|>system
You are a helpful interview preparation coach.<|im_end|>
<|im_start|>user
Give me a step-by-step preparation plan for Research Engineer interviews at frontier AI labs.<|im_end|>
<|im_start|>assistant
Preparing for Research Engineer interviews at Frontier AI Labs can be broken down into several key steps to ensure you're well-prepared and confident during the process. Here’s a step-by-step guide:

### 1. Understand the Company Culture
- **Research**: Familiarize yourself with Frontier AI Labs' mission, values, and culture. This will help you understand how your skills fit within their team.
  
### 2. Prepare Your Technical Background
- **Technical Skills**: Review your technical background and expertise in areas such as machine learning, deep learning, natural language processing, computer vision, etc.
  
- **Projects**:
   - Identify projects that showcase your technical skills and contributions.
   - Highlight specific achievements or challenges you faced and overca

### Implement Different Inference strategies from Scratch
1. Greedy Decoding
2. Top-k sampling
3. Top-p sampling
4. Beam-Search
5. Speculative decoding